# Microsoft Fabric Notebook: 01_ingest_customers
**Target Lakehouse**: `Bronze_Lakehouse`  
**Target Table**: `raw_customers` (Delta Lake)  
**Description**: Ingests raw customer CSV datasets into Bronze_Lakehouse and appends operational lineage fields.


In [ ]:
# Fabric Notebook Parameters Cell (Configurable via Fabric Data Factory Pipelines)
pipeline_run_id = "RUN_FABRIC_20260908"
environment = "PROD"
source_system = "FABRIC_INGEST_ENGINE"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, input_file_name

# Read Raw CSV from OneLake Bronze Landing Files
source_path = "abfss://ECommerce-Lakehouse@onelake.dfs.fabric.microsoft.com/Bronze_Lakehouse.Lakehouse/Files/raw_customers.csv"
raw_df = spark.read.option("header", "true").option("inferSchema", "true").csv(source_path)

# Enrich with Operational Ingestion Metadata
bronze_df = raw_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name()) \
    .withColumn("batch_id", lit("BATCH_20260908")) \
    .withColumn("pipeline_run_id", lit(pipeline_run_id)) \
    .withColumn("source_system", lit(source_system))

# Save as Delta Table in Bronze Lakehouse
bronze_df.write.format("delta").mode("overwrite").saveAsTable("raw_customers")
print(f"[FABRIC BRONZE] Successfully ingested {bronze_df.count()} records into raw_customers.")
